# SRILM In-Class Activity - worked solution (STUDY GUIDE)
Practice material for exam preparation. Re-derive and re-run everything yourself before relying on it.
**Setup (Colab):** upload the course zip (browndata/, ngram, ngram-count) and run the next cell.

In [ ]:
import os, subprocess
if os.path.exists('/content'):                      # Colab only
    from google.colab import files
    up = files.upload()                             # choose the course .zip
    for n in up: subprocess.run(['unzip','-o','-q',n])
for b in ['ngram','ngram-count']:
    if os.path.exists(b): os.chmod(b, 0o755)
print(subprocess.run('./ngram-count -help | head -3', shell=True, capture_output=True, text=True).stdout)

## Part A - Toy example (LM-InClass-Activity): counts and MLE

In [ ]:
from collections import Counter
D = ["<s> I am Sam </s>","<s> Sam I am </s>","<s> Sam I like </s>","<s> Sam I do like </s>","<s> do I like Sam </s>"]
uni, bi = Counter(), Counter()
for s in D:
    t = s.split(); uni.update(t); bi.update(zip(t, t[1:]))
N = sum(uni.values()) - uni["<s>"]                  # predicted tokens (<s> is never predicted)
def p_uni(w, add=0, V=0): return (uni[w] + add) / (N + add * V)
def p_bi(w, prev, add=0, V=0): return (bi[(prev, w)] + add) / (uni[prev] + add * V)
def sent(s, kind, **kw):
    t = s.split(); p = 1.0; parts = []
    for a, b in zip(t, t[1:]):
        q = p_bi(b, a, **kw) if kind == "bi" else p_uni(b, **kw)
        parts.append((b, q)); p *= q
    return p, parts
S1, S2 = "<s> Sam I do I like </s>", "<s> Sam I am </s>"
for k in ("bi", "uni"):
    print(k, "S1 =", sent(S1, k)[0], " S2 =", sent(S2, k)[0], "-> better:", "S2" if sent(S2, k)[0] > sent(S1, k)[0] else "S1")

**Q1/Q2 answer:** S2 (`<s> Sam I am </s>`) wins under both models (bigram 0.072 vs 0.0096; unigram 1.07e-3 vs 3.3e-5).
Reason: S1 contains the rare transitions I->do (1/5) and do->I (1/2) and is longer; more factors < 1 multiply to a smaller probability.

In [ ]:
S3 = "<s> Sam I do like linguistics </s>"
print("MLE unigram:", sent(S3, "uni")[0])

In [ ]:
t = S3.split()
print("bigram MLE per word:")
for a, b in zip(t, t[1:]):
    print(f"  P({b}|{a}) = {bi[(a,b)]}/{uni[a]}" + (f" = {p_bi(b,a):.3f}" if uni[a] else " (undefined, context unseen)"))
print("=> sentence probability is 0 (unseen word 'linguistics').")
V = len(uni) + 1                                   # + linguistics
print("add-one:", sent(S3, "uni", add=1, V=V)[0], sent(S3, "bi", add=1, V=V)[0])

**Q3 answer:** every term is non-zero except `linguistics` (count 0), so unsmoothed MLE gives probability 0 for the sentence in both models. Add-one (or any smoothing / `<unk>`) gives a small positive value.

## Part B - SRILM on the Brown corpus (steps 1-3 of the handout)

In [ ]:
import subprocess, re, os, json
BIN = "."; W = "work"; os.makedirs(W, exist_ok=True)
def sh(c): r = subprocess.run(c, shell=True, capture_output=True, text=True, cwd=BIN); return r.stdout + r.stderr
print(sh("./ngram-count -text browndata/brown-train.txt -order 2 -write work/brown-train.count"))
print(sh("head -5 work/brown-train.count; grep -c . work/brown-train.count"))
# cross-check a count with UNIX tools:  'the' occurrences (exact token)
print(sh("tr ' ' '\\n' < browndata/brown-train.txt | grep -c '^the$'"), sh("grep -P '^the\\t' work/brown-train.count"))

In [ ]:
import subprocess, re, os, json, sys
BIN = "."
W = "work"; os.makedirs(os.path.join(BIN, W), exist_ok=True)
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True, cwd=BIN); return r.stdout + r.stderr
def train(text, order, name, opts):
    sh(f"./ngram-count -text {text} -order {order} -write {W}/{name}.count -lm {W}/{name}.lm {opts}")
    return f"{W}/{name}.lm"
def ppl(lm, text, extra=""):
    o = sh(f"./ngram -lm {lm} -ppl {text} {extra}")
    m = re.search(r"(\d+) OOVs", o); z = re.search(r"(\d+) zeroprobs, logprob= (\S+) ppl= (\S+) ppl1= (\S+)", o)
    return dict(oov=int(m.group(1)), zero=int(z.group(1)), logprob=float(z.group(2)), ppl=float(z.group(3)), ppl1=float(z.group(4)))
TR, DEV, TE = "browndata/brown-train.txt", "browndata/brown-dev.txt", "browndata/brown-test.txt"

In [ ]:
R = {}
# Q1: unsmoothed bigram
R["q1_unk_add0"] = ppl(train(TR, 2, "bi_add0", "-unk -addsmooth 0"), TE)
R["q1_nounk_add0"] = ppl(train(TR, 2, "bi_add0_nounk", "-addsmooth 0"), TE)
# Q2: other smoothing (bigram, -unk)
for k, o in {"gt": "", "wb": "-wbdiscount -interpolate", "kn": "-kndiscount -interpolate", "ukn": "-ukndiscount -interpolate",
             "add1": "-addsmooth 1", "add0.1": "-addsmooth 0.1"}.items():
    R["q2_" + k] = ppl(train(TR, 2, "bi_" + k, "-unk " + o), TE)
# Q3: trigram
for k, o in {"add0": "-addsmooth 0", "gt": "", "kn": "-kndiscount -interpolate"}.items():
    R["q3_tri_" + k] = ppl(train(TR, 3, "tri_" + k, "-unk " + o), TE)
# Q4: preprocessing
STOP = set("a an the of and or to in on at for with by is are was were be been it its this that these those i he she we they you as".split())
def prep(src, dst, punct, stop, lower):
    with open(os.path.join(BIN, src)) as f, open(os.path.join(BIN, dst), "w") as g:
        for l in f:
            if lower: l = l.lower()
            t = l.split()
            if punct: t = [w for w in t if re.search(r"\w", w)]
            if stop: t = [w for w in t if w.lower() not in STOP]
            if t: g.write(" ".join(t) + "\n")
for k, (p, s, lo) in {"punct": (1, 0, 1), "stop": (0, 1, 0), "both": (1, 1, 1)}.items():
    prep(TR, f"{W}/tr_{k}.txt", p, s, lo); prep(TE, f"{W}/te_{k}.txt", p, s, lo)
    R["q4_" + k] = ppl(train(f"{W}/tr_{k}.txt", 2, "p_" + k, "-unk -kndiscount -interpolate"), f"{W}/te_{k}.txt")
R["q4_base_kn"] = R["q2_kn"]
# Q5: mix of unigram + bigram, tune (addsmooth, lambda) on dev
grid = {}
for a in [0.001, 0.01, 0.1, 0.5]:
    uni = train(TR, 1, f"u_{a}", f"-unk -addsmooth {a}"); bi = train(TR, 2, f"b_{a}", f"-unk -addsmooth {a}")
    for lam in [0.1, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0]:
        grid[f"{a}|{lam}"] = ppl(bi, DEV, f"-mix-lm {uni} -lambda {lam}")["ppl"]
best = min(grid, key=grid.get); ba, bl = best.split("|")
uni = train(TR, 1, f"u_{ba}", f"-unk -addsmooth {ba}"); bi = train(TR, 2, f"b_{ba}", f"-unk -addsmooth {ba}")
R["q5_grid"] = grid; R["q5_best"] = dict(addsmooth=float(ba), lam=float(bl), dev_ppl=grid[best],
                                         test=ppl(bi, TE, f"-mix-lm {uni} -lambda {bl}"))

## Part C - Answers to the exercises (numbers from the run above)
1. **-inf / OOV.** With `-addsmooth 0` every word's probability is its relative frequency, and `<unk>` gets count 0, so each OOV word has P = 0 and log P = -inf (39,856 zeroprobs on the test set; ppl 6270.9 only because zero-probability events are excluded from the perplexity). `-unk` keeps `<unk>` in the vocabulary so unseen test words are mapped to it; `-addsmooth 0` means add-0 (no smoothing). Fix: use real smoothing (Good-Turing/Kneser-Ney/Witten-Bell) or add-delta>0 so `<unk>` gets mass.
2. **Smoothing (bigram, test ppl):** add-1 2493, add-0.1 1209, Good-Turing 642, Kneser-Ney 618, Witten-Bell 416, modified KN 362; zero probabilities drop to 0 for all. Add-delta is poor on large vocabularies; discounting methods are far better.
3. **Trigram** (KN): 587 vs 618 for bigram; unsmoothed trigram still has -inf. Higher order helps a little but data sparsity grows.
4. **Preprocessing:** perplexities are NOT comparable across rows, because removing tokens changes the test text itself (fewer, harder-to-predict tokens give higher per-word ppl).
5. **Interpolation:** best on dev is addsmooth = 0.001, lambda = 0.7 (dev ppl 581.0); test ppl 681.3. SRILM's `-lambda` weights the *main* (`-lm`, bigram) model; `-mix-lm` is the unigram.